# Load Dependencies


## Files needed to run:

- `feature_columns.pkl`
- `svm_finetuned.pkl`
- `rf_finetuned.pkl`
- `CBERT_checkpoint.pth`
- `EnsembleProcessing.py`
- All files are available in the Final Implementation folder of our [repository](https://github.com/MiguelPartosa/Thesis-FOS-BinaryClass-WSD).


In [ ]:
import pandas as pd
import torch
from torch import cuda
from transformers import DistilBertTokenizer, DistilBertModel

c:\Users\Miguel\Documents\Project Source Files\IT Work\School\Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## CBERT Checkpoint


In [ ]:
device = 'cuda' if cuda.is_available() else 'cpu'


class BertClass(torch.nn.Module):
    def __init__(self):
        super(BertClass, self).__init__()
        self.l1 = DistilBertModel.from_pretrained('GianTan/CBERTo')
        self.pre_classifier = torch.nn.Linear(768, 768)
        self.dropout = torch.nn.Dropout(0.3)

        self.pre_classifier2 = torch.nn.Linear(768, 768)
        self.dropout2 = torch.nn.Dropout(0.3)

        self.classifier = torch.nn.Linear(768, 1)

    def forward(self, input_ids, attention_mask, token_type_ids):
        output_1 = self.l1(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = output_1[0]
        pooler = hidden_state[:, 0]
        pooler = self.pre_classifier(pooler)
        pooler = torch.nn.Tanh()(pooler)
        pooler = self.dropout(pooler)
        pooler = self.pre_classifier2(pooler)
        pooler = torch.nn.Tanh()(pooler)
        pooler = self.dropout2(pooler)

        output = self.classifier(pooler)
        return output.squeeze(1)


cbert_model = BertClass()
cbert_model.to(device)

tokenizer = DistilBertTokenizer.from_pretrained(
    'GianTan/CBERTo', truncation=True, do_lower_case=False)
optimizer = torch.optim.Adam(params=cbert_model.parameters(), lr=4e-05)

# load
checkpoint = torch.load('CBERT_checkpoint.pth', map_location='cpu')
cbert_model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

## SVM and Random Forest Checkpoint


Importing


In [ ]:
import pickle
import pandas as pd
from EnsembleProcessing import process_embeddings
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

# Dependencies from read pickle file


def GetTextCol(X):
    text_cols = X.select_dtypes(include=['object', 'string']).columns
    if len(text_cols) == 0:
        raise ValueError("No text columns found in input DataFrame")
    # Error is raised if the first index is not returned.
    return text_cols[0]


def GetNumCol(X):
    return X.select_dtypes(include=['int64', 'float64']).columns.tolist()


preprocessor = ColumnTransformer(
    transformers=[
        ("tfidf", TfidfVectorizer(), GetTextCol),
        ("scaler", StandardScaler(), GetNumCol)
    ],
    remainder='drop'  # Remove any unhandled columns
)

# SVM
with open('svm_finetuned.pkl', 'rb') as f:
    svm_best = pickle.load(f)

# Random Forest
with open('rf_finetuned.pkl', 'rb') as f:
    rf_best = pickle.load(f)

# Model Function Backend


## CBERT


In [ ]:
def test_model(item):
    input_text = item
    encoded_text = tokenizer.encode_plus(
        input_text,
        None,
        add_special_tokens=True,
        max_length=256,
        pad_to_max_length=True,
        return_token_type_ids=True
    )

    # Convert the input to tensors
    input_ids = torch.tensor(encoded_text['input_ids']).unsqueeze(0)
    input_mask = torch.tensor(encoded_text['attention_mask']).unsqueeze(0)
    segment_ids = torch.tensor(encoded_text['token_type_ids']).unsqueeze(0)

    # Move tensors to the device
    input_ids = input_ids.to(device)
    input_mask = input_mask.to(device)
    segment_ids = segment_ids.to(device)

    # Make predictions
    with torch.no_grad():
        outputs = cbert_model(input_ids, input_mask, segment_ids)

    # Apply sigmoid activation function
    outputs = torch.sigmoid(outputs)

    # Convert the outputs to numpy array
    outputs = outputs.cpu().detach().numpy()

    return outputs

## SVM and Random Forest


In [ ]:
# When predicting on new data:
def PredictSample(input_df):
    # Load models and feature columns
    with open('svm_finetuned.pkl', 'rb') as f:
        svm_model = pickle.load(f)

    # Random Forest
    with open('rf_finetuned.pkl', 'rb') as f:
        rf_model = pickle.load(f)

    with open('feature_columns.pkl', 'rb') as f:
        feature_columns = pickle.load(f)

    try:
        # Embeddings
        embeddings_df = process_embeddings(input_df, variance_threshold=1)
        combined_df = pd.concat([embeddings_df, input_df], axis=1)

        # Clean read issue tensor values
        if 'Similarity Scores' in combined_df.columns:
            combined_df['Similarity Scores'] = combined_df['Similarity Scores'].apply(
                lambda x: x.item() if hasattr(x, 'item') else x
            )

        # Drop Non-features
        for col in ['Is FOS', 'Word Sense', 'Verb', 'Usage']:
            if col in combined_df.columns:
                combined_df = combined_df.drop(columns=[col])

        # Align
        # Create a DataFrame with all required columns, filled with zeros
        aligned_row = pd.DataFrame(0, index=[0], columns=feature_columns)

        for col in combined_df.columns:
            if col in feature_columns:
                aligned_row[col] = combined_df[col]

        rf_prob = rf_model.predict_proba(aligned_row)[0][1]
        svm_pred = svm_model.predict(aligned_row)[0]
        return {
            # 'rf_probability': rf_prob,
            'rf_prediction': 1 if rf_prob >= 0.5 else 0,
            'svm_prediction': int(svm_pred)
        }

    except Exception as e:
        # print(f"Error during prediction: {e}")
        print(e)
        # return {'error': str(e)}

In [ ]:
import warnings


def prediction_values(score):
    return 'Non-Literal' if score >= 0.5 else 'Literal'

# trained on "is fos" column


def ClassifyExample(df) -> tuple:
    '''
    Returns Cbert, SVM, and RF
    '''
    cbert_result = prediction_values(test_model(df['Usage'][0]))
    with warnings.catch_warnings(action="ignore"):
        models_result = PredictSample(df)
    svm_result = prediction_values(models_result['svm_prediction'])
    rf_result = prediction_values(models_result['rf_prediction'])
    return (cbert_result, svm_result, rf_result)

# Test Models


In [ ]:
# TODO Params
# @title Enter new parameters and rerun cell to refresh results. { display-mode: "form" }
from IPython.display import HTML, display, Javascript

test_fos = 'makabuhi og patay'  # @param {type:"string"}
test_word_sense = 'usa ka makapatikod nga bakak o sugilanon' # @param {type:"string"}
test_verb = 'Makabuhi, Patay'  # @param {type:"string"}

test_usage = 'Ang mga tawo nga makabuhi og patay kasagaran maayo kaayo mamatay.' # @param {type:"string"}
test_is_fos = 'Non-literal Example' # @param ["Non-literal Example", "Literal Example"]

test_is_fos = 0 if test_is_fos == "Literal Example" else 1

test_df = pd.DataFrame({'FOS': [test_fos], 'Word Sense': [test_word_sense], 'Verb': [
                       test_verb], 'Usage': [test_usage], 'Is FOS': [test_is_fos]})

prediction_output = ClassifyExample(test_df)
print('\n\n')
df = pd.DataFrame({
    "Model": ["Cbert", "SVM", "Random Forest"],
    "Prediction": prediction_output
})

html = f"""
            <div style="scale: 100%;">
                {df.to_html(index=False, border=1)}
            </div>
        """

display(HTML(html))
Javascript("google.colab.output.setIframeHeight('500px');")

c:\Users\Miguel\Documents\Project Source Files\IT Work\School\Thesis\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:2834: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


Generating embeddings...


Transforming Usage Embeddings: 100%|██████████| 3/3 [00:00<00:00, 87.96it/s]


Number of components for 100% variance:
Verb: 768 components
Usage: 768 components
Sentence: 768 components
Optimal number of clusters:
Verb: 2 clusters
Usage: 2 clusters
Sentence: 2 clusters





Model,Prediction
Cbert,Non-Literal
SVM,Non-Literal
Random Forest,Literal


<IPython.core.display.Javascript object>